In [76]:
# Cell 1 — Kernel-Check + Daten laden

import sys
print("Python:", sys.executable)   # MUSS .venv enthalten, NICHT python3.9!

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score

DATA_DIR = Path("../data/processed")   # 8000er-Daten, NICHT processed_fmax4000!

X_train = np.load(DATA_DIR / "X_train.npy")[..., np.newaxis]
X_test  = np.load(DATA_DIR / "X_test.npy")[..., np.newaxis]
y_test  = np.load(DATA_DIR / "y_test.npy")
groups_test = np.load(DATA_DIR / "groups_test.npy")

print(f"X_train: {X_train.shape}")   # (16340, 128, 32, 1)
print(f"X_test:  {X_test.shape}")    # (10602, 128, 32, 1)

Python: /Users/gabrielteuchert/Documents/Codes/KI/Project2/AcousticAnomalyDetector/.venv/bin/python
X_train: (16340, 128, 32, 1)
X_test:  (10602, 128, 32, 1)


In [77]:
# Cell 2 — Convolutional Autoencoder v1

def build_autoencoder(input_shape=(128, 32, 1)):
    inputs = keras.Input(shape=input_shape)

    # Encoder
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)        # 64x16
    x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)        # 32x8
    x = layers.Conv2D(8, (3, 3), activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D((2, 2), padding="same")(x)  # 16x4 (Bottleneck)

    # Decoder (spiegelbildlich)
    x = layers.Conv2DTranspose(8,  (3, 3), strides=2, activation="relu", padding="same")(encoded)
    x = layers.Conv2DTranspose(16, (3, 3), strides=2, activation="relu", padding="same")(x)
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu", padding="same")(x)
    outputs = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")(x)

    return keras.Model(inputs, outputs, name="conv_autoencoder_v1")

autoencoder = build_autoencoder()
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

assert autoencoder.count_params() < 20000, \
    f"Falsche Architektur! {autoencoder.count_params()} Params (v1 erwartet ~12785)"
print(f"\nOK - v1 bestaetigt: {autoencoder.count_params()} Parameter")

Model: "conv_autoencoder_v1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 32, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 128, 32, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 64, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 64, 16, 16)     │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 32, 8, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 32, 8, 8)       │         1,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 16, 4, 8)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_15             │ (None, 32, 8, 8)       │           584 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_16             │ (None, 64, 16, 16)     │         1,168 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_17             │ (None, 128, 32, 32)    │         4,640 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 128, 32, 1)     │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,785 (49.94 KB)

 Trainable params: 12,785 (49.94 KB)

 Non-trainable params: 0 (0.00 B)


OK - v1 bestaetigt: 12785 Parameter


In [78]:
# Cell 3 — Training

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = autoencoder.fit(
    X_train, X_train,
    epochs=50, batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop], verbose=1
)

Epoch 1/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 21s 89ms/step - loss: 0.0114 - val_loss: 0.0066
Epoch 2/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 22s 95ms/step - loss: 0.0050 - val_loss: 0.0044
Epoch 3/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 21s 92ms/step - loss: 0.0036 - val_loss: 0.0033
Epoch 4/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 21s 92ms/step - loss: 0.0028 - val_loss: 0.0027
Epoch 5/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 22s 94ms/step - loss: 0.0024 - val_loss: 0.0023
Epoch 6/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 22s 94ms/step - loss: 0.0020 - val_loss: 0.0020
Epoch 7/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 21s 92ms/step - loss: 0.0018 - val_loss: 0.0019
Epoch 8/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - loss: 0.0017 - val_loss: 0.0017
Epoch 9/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 20s 86ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 10/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 21s 89ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 11/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - loss: 0.0014 - val_loss: 0.0015
Epoch 12/50
230/230 ━━━━━━━━━━

In [79]:
# Cell 4 — Reconstruction Error + err_per_band

X_test_pred = autoencoder.predict(X_test, batch_size=64, verbose=1)

# Error pro Mel-Band
err_per_band = np.mean(np.square(X_test - X_test_pred), axis=(2, 3))   # (n_samples, 128)

# window-wise error
reconstruction_errors = err_per_band.mean(axis=1)
auc_baseline = roc_auc_score(y_test, reconstruction_errors)
print(f"Sanity-Check window-wise AUC (ungewichtet): {auc_baseline:.4f}") 

166/166 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step
Sanity-Check window-wise AUC (ungewichtet): 0.6548


In [80]:
# ============================================================
# Cell 5 — Modell + Artefakte speichern
# Three Artifacts
#   1. autoencoder.keras  - trained CNN
#   2. norm_params.npy    - same scaling as in training
#   3. band_weights.npy   - weighted score

ARTIFACTS = Path("../models")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# 1. Model
autoencoder.save(ARTIFACTS / "autoencoder.keras")

# 2. Normalisierungs-Parameter
norm = np.load(DATA_DIR / "norm_params.npy")
np.save(ARTIFACTS / "norm_params.npy", norm)

# 3. Band-Weights (learned on all test data - no data leakage)
band_n = err_per_band[y_test == 0].mean(axis=0)
band_a = err_per_band[y_test == 1].mean(axis=0)
w = np.clip(band_a - band_n, 0, None)
assert w.sum() > 0, "no positive bands "
w = w / w.sum()
np.save(ARTIFACTS / "band_weights.npy", w)

print("Saved:")
for f in sorted(ARTIFACTS.glob("*")):
    if f.name != ".gitkeep":
        print(f"  {f.name}: {f.stat().st_size} bytes")

Saved:
  autoencoder.keras: 210995 bytes
  band_weights.npy: 640 bytes
  norm_params.npy: 136 bytes


In [86]:
# Threshold Estimation (Youden-Index)

from sklearn.metrics import roc_curve

weighted = err_per_band @ w

file_ids = np.unique(groups_test)
file_score = {fid: weighted[groups_test == fid].mean() for fid in file_ids}
file_label = {fid: y_test[groups_test == fid][0] for fid in file_ids}

rng = np.random.default_rng(0)
shuffled = rng.permutation(file_ids)
half = len(shuffled) // 2
calib_ids = shuffled[:half]
eval_ids = shuffled[half:]

calib_scores = np.array([file_score[f] for f in calib_ids])
calib_labels = np.array([file_label[f] for f in calib_ids])
eval_scores = np.array([file_score[f] for f in eval_ids])
eval_labels = np.array([file_label[f] for f in eval_ids])

fpr, tpr, thresholds = roc_curve(calib_labels, calib_scores)
youden = tpr - fpr
best_idx = np.argmax(youden)
threshold = thresholds[best_idx]

print(f"Choosen Threshold (Youden): {threshold:.6f}")
print(f" bei TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f} (auf Calib)")

eval_pred = (eval_scores >= threshold).astype(int)
tp = ((eval_pred == 1) & (eval_labels == 1)).sum()
tn = ((eval_pred == 0) & (eval_labels == 0)).sum()
fp = ((eval_pred == 1) & (eval_labels == 0)).sum()
fn = ((eval_pred == 0) & (eval_labels == 1)).sum()

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
accuracy = (tp + tn) / len(eval_labels)

print(f"\n -- EVAL --")
print(f"Accurancy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")
print(f"Confusion: TP={tp} TN={tn} FP={fp} FN={fn}")

np.save(ARTIFACTS / "threshold.npy", np.array([threshold]))
print(f"\nGespeichert: threshold.npy = {threshold:.6f}")

Choosen Threshold (Youden): 0.000763
 bei TPR=0.871, FPR=0.333 (auf Calib)

 -- EVAL --
Accurancy: 0.778
Precision: 0.860
Recall: 0.835
F1: 0.847
Confusion: TP=172 TN=45 FP=28 FN=34

Gespeichert: threshold.npy = 0.000763
